# CrossFuse v5 - Stage D: Ablations

> **Note (2026-09):** this grid scores **in-domain FakeAVCeleb only**, where nearly every arm lands near 0.99, so it cannot show *which change fixed cross-dataset transfer*. That question was answered by `D2-ablation-attribution.ipynb` (A4/A13/A14/A15 scored on Celeb-DF + DFDC). This 16-row grid has **not** been run against the CLIP configuration. Known issue: `A16_clip_full_finetune` is not real full fine-tuning (ViTs are LayerNorm-only in `set_backbone_trainable`), so it duplicates `A4` -- drop or relabel it. Rows `A1`-`A12` are the paper's ablations (`results/ablation_results_v5.csv`).

One seed, reduced epoch budget (8 fine-tune epochs instead of 20), across the
full grid.

Rows `A13`-`A16` isolate the two changes made after the first v5 report - the
CLIP backbone and the FF++ training mixture - so each can be credited or
dismissed separately rather than as one bundled "it got better".

`A13_no_ffpp_data` and `A14_efficientnet_backbone` are the two that matter: if
the improvement is entirely `A14`, the data change did nothing and the story is
about the backbone, and vice versa.

Writes `/kaggle/working/ablation_results_v5.csv` incrementally, so a crash
partway through loses at most the row in progress. Budget: ~3h on one seed for
the EfficientNet rows; the CLIP rows are slower, so consider running the grid
in two sessions.

In [ ]:
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, copy, json
from collections import defaultdict
import numpy as np
import torch

from crossfuse_v5 import (
    CONFIG, run_training_pipeline, evaluate_deterministic, safe_auc,
    roc_curve, balanced_accuracy_score, merge_manifests,
)

# ---- FakeAVCeleb ----
CROP_DIR = None
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        head = f.readline()
    if "audio_label" in head:
        CROP_DIR = os.path.dirname(p)
        break
assert CROP_DIR, "FakeAVCeleb manifest.csv not found -- attach crops-v5."
with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    FAV_ROWS = list(csv.DictReader(f))
for r in FAV_ROWS:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])

# ---- FF++ ----
FFPP_DIR, FFPP_ROWS = None, []
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        head = f.readline()
    if "method" in head and "target_id" in head:
        FFPP_DIR = os.path.dirname(p)
        break
if FFPP_DIR:
    with open(os.path.join(FFPP_DIR, "manifest.csv")) as f:
        FFPP_ROWS = list(csv.DictReader(f))
    for r in FFPP_ROWS:
        r["label"] = int(r["label"])

ENC = next(iter(glob.glob("/kaggle/input/**/ffpp_encoder_v5.pth", recursive=True)), None)
print(f"FakeAVCeleb {len(FAV_ROWS)} rows | FF++ {len(FFPP_ROWS)} rows | "
      f"encoder {'yes' if ENC else 'NO'}")

# Two manifests, so a row can drop FF++ data while holding everything else fixed.
MANIFEST_MERGED = (merge_manifests(FAV_ROWS, CROP_DIR, FFPP_ROWS, FFPP_DIR,
                                   seed=CONFIG["SEED"], verbose=False)
                   if FFPP_ROWS else None)
MANIFEST_FAV = [dict(r, crop_dir=CROP_DIR, has_audio=1) for r in FAV_ROWS]

N_ABLATION_SEEDS = 1
ABLATION_EPOCHS = dict(EPOCHS_FROZEN=2, EPOCHS_FINETUNE=8, PATIENCE=4)


def cfg_variant(**overrides):
    c = copy.deepcopy(CONFIG)
    c.update(ABLATION_EPOCHS)
    c.update(overrides)
    return c


def quick_youden(y_true, y_prob, default=0.5):
    y_true = np.asarray(y_true)
    if len(set(y_true.astype(int).tolist())) < 2:
        return default
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])


def quick_eval_row(result, cfg):
    val = evaluate_deterministic(result["model"], result["val_loader"], cfg)
    test = evaluate_deterministic(result["model"], result["test_loader"], cfg)
    row = {"val_video_auc": val["video_auc"], "val_audio_auc": val["audio_auc"],
          "val_sync_auc": val["sync_auc"], "val_fusion_auc": val["fusion_auc"],
          "test_video_auc": test["video_auc"], "test_audio_auc": test["audio_auc"],
          "test_sync_auc": test["sync_auc"], "test_fusion_auc": test["fusion_auc"],
          "n_test": len(result["test_idx"])}
    for head, k in (("video", "v"), ("audio", "a"), ("fusion", "f"), ("sync", "s")):
        vt, vp = val["raw"].get(f"{k}_true"), val["raw"].get(f"{k}_prob")
        tt, tp = test["raw"].get(f"{k}_true"), test["raw"].get(f"{k}_prob")
        if not vt or not tt:
            row[f"{head}_bacc"] = float("nan")
            continue
        thr = quick_youden(vt, vp)
        pred = (np.asarray(tp) >= thr).astype(int)
        row[f"{head}_bacc"] = (balanced_accuracy_score(tt, pred)
                               if len(set(np.asarray(tt).astype(int).tolist())) > 1
                               else float("nan"))
    return row

In [ ]:
# Every row is (name, cfg, manifest). The manifest is part of the arm because
# "with vs without FF++ data" is a DATA ablation, not a config flag.
BASE = dict(BACKBONE="clip_vit_l14", TRAIN_RES=224,
            PRETRAINED_ENCODER=ENC, BATCH_CLIPS=4, GRAD_ACCUM=3)
M = MANIFEST_MERGED if MANIFEST_MERGED else MANIFEST_FAV

ABLATIONS = [
    ("A1_video_only", cfg_variant(**BASE, MODALITY="video_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False), M),
    ("A2_audio_only", cfg_variant(**BASE, MODALITY="audio_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False), M),
    ("A3_two_head_no_fusion", cfg_variant(**BASE, USE_CROSS_ATTENTION=False,
                                          USE_SYNC_HEAD=False, USE_FUSION_HEAD=False), M),
    ("A4_full_crossfuse", cfg_variant(**BASE), M),          # matches NB-B's main config
    ("A5_no_hard_negatives", cfg_variant(**BASE, GENERATE_HARD_NEGATIVES=False), M),
    ("A6_no_compression_aug", cfg_variant(**BASE, USE_COMPRESSION_AUG=False), M),
    ("A7_no_mfcc_cmvn", cfg_variant(**BASE, USE_MFCC_CMVN=False), M),
    ("A8_random_split", cfg_variant(**BASE, IDENTITY_DISJOINT_SPLIT=False), M),
    ("A9_late_fusion_baseline", cfg_variant(**BASE, LATE_FUSION_BASELINE=True), M),
    ("A10_sync_unaligned", cfg_variant(**BASE, SYNC_DENSE_ALIGNED=False, SYNC_ARCH="pooled_ca"), M),
    ("A11_no_paired_batches", cfg_variant(**BASE, USE_PAIRED_BATCHES=False), M),
    ("A12_no_silence_trim", cfg_variant(**BASE, AUDIO_LEAD_SKIP=0.0), M),

    # ---- The four rows that isolate the post-v5 changes ----
    # Drop FF++ from the mixture, keep the FF++-pretrained CLIP encoder.
    ("A13_no_ffpp_data", cfg_variant(**BASE), MANIFEST_FAV),
    # Revert the backbone, keep the FF++ mixture. No encoder: the checkpoint is
    # a ViT and would not load into EfficientNet (load_visual_encoder raises).
    ("A14_efficientnet_backbone",
     cfg_variant(BACKBONE="efficientnet_b4", TRAIN_RES=224, PRETRAINED_ENCODER=None,
                 FREEZE_BLOCKS=4), M),
    # CLIP + FF++ mixture but WITHOUT the NB-B1 pretraining stage: isolates how
    # much came from the dedicated FF++ pretrain vs. simply mixing FF++ in.
    ("A15_no_ffpp_pretrain", cfg_variant(**dict(BASE, PRETRAINED_ENCODER=None)), M),
    # Full fine-tune instead of LN-only. LNCLIP-DF reports this OVERFITS badly;
    # this row is here to confirm that on our data rather than assume it.
    ("A16_clip_full_finetune",
     cfg_variant(**BASE, FREEZE_BLOCKS=0, LR_LN=None, LR_BACKBONE=1e-5), M),
]

if not MANIFEST_MERGED:
    print("[!] FF++ crops not attached -- A13 is identical to A4 and the data "
          "ablation is meaningless. Attach ffpp-crops-v5.")
if ENC is None:
    print("[!] No FF++ encoder attached -- A15 is identical to A4. Run NB-B1.")

print(f"Running {len(ABLATIONS)} ablation configs x {N_ABLATION_SEEDS} seed(s)...")

In [ ]:
ablation_csv_path = "/kaggle/working/ablation_results_v5.csv"
fieldnames = ["row", "seed", "backbone", "n_ffpp", "encoder", "n_test",
              "val_video_auc", "val_audio_auc", "val_sync_auc", "val_fusion_auc",
              "test_video_auc", "test_audio_auc", "test_sync_auc", "test_fusion_auc",
              "video_bacc", "audio_bacc", "fusion_bacc", "sync_bacc"]


def _write_ablation_csv(rows):
    with open(ablation_csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


ablation_rows = []
for name, cfg, manifest in ABLATIONS:
    for si in range(N_ABLATION_SEEDS):
        seed = CONFIG["SEED"] + si
        print(f"\n{'=' * 60}\n  {name}  (seed {seed})\n{'=' * 60}")
        result = run_training_pipeline(manifest, CROP_DIR, cfg, tag=name, seed=seed,
                                       verbose=True)
        row = quick_eval_row(result, cfg)
        row.update(row=name, seed=seed, backbone=cfg["BACKBONE"],
                   n_ffpp=sum(1 for r in manifest if r.get("corpus") == "ffpp"),
                   encoder=os.path.basename(cfg["PRETRAINED_ENCODER"] or "none"))
        ablation_rows.append(row)
        _write_ablation_csv(ablation_rows)
        print(f"{name} (seed {seed}) -> video={row['test_video_auc']:.4f} "
              f"audio={row['test_audio_auc']:.4f} fusion={row['test_fusion_auc']:.4f} "
              f"sync={row['test_sync_auc']:.4f}")
        del result["model"]
        torch.cuda.empty_cache()

print(f"\nAll ablations complete. Written to {ablation_csv_path}")